# Customer Churn Prediction — Data Preprocessing

## Objective

The objective of this notebook is to prepare the customer dataset for machine learning.

The preprocessing process includes feature selection, train-test splitting, categorical encoding, numerical scaling, and the construction of a reproducible preprocessing pipeline while preventing data leakage.

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [22]:
df = pd.read_csv(
    "../data/raw/sales.csv"
)

## 1. Initial Data Preparation

Before splitting and transforming the dataset, the data quality issues identified during the exploratory analysis are addressed.

The `customerID` variable is retained separately for customer identification but excluded from the predictive features, since the identifier itself does not contain meaningful information for predicting churn.

In [23]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [24]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [25]:
df["TotalCharges"].isna().sum()

np.int64(0)

In [26]:
df["TotalCharges"].dtype

dtype('float64')

In [27]:
customer_ids = df["customerID"]

X = df.drop(columns=["customerID", "Churn"])

y = df["Churn"]

In [28]:
y = y.map({
    "No": 0,
    "Yes": 1
})

In [29]:
y.value_counts()
y.value_counts(normalize=True) * 100

Churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64

## 2. Train-Test Split

The dataset is divided into training and test sets using an 80/20 split.

Stratified sampling is applied to preserve the original churn distribution in both subsets. Approximately 26.5% of customers churn in both the training and test sets.

A fixed random state is used to ensure that the split is reproducible.

The test set will remain unseen during preprocessing decisions and model training and will only be used for final model evaluation.

In [30]:
from sklearn.model_selection import train_test_split

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [32]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(5634, 19)
(1409, 19)
(5634,)
(1409,)


In [33]:
print(y_train.mean())
print(y_test.mean())

0.2653532126375577
0.2654364797728886


In [34]:
X_train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
3738,Male,0,No,No,35,No,No phone service,DSL,No,No,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65
3151,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55
4860,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,Yes,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35
3867,Female,0,Yes,No,26,Yes,No,DSL,No,Yes,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.70
3810,Male,0,Yes,Yes,1,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,No,Electronic check,44.55,44.55


In [35]:
numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod",
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]

## 3. Preprocessing Pipeline

A preprocessing pipeline is constructed to ensure that all transformations are applied consistently and without data leakage.

Numerical features are standardized using `StandardScaler`, while categorical variables are encoded using `OneHotEncoder`. Binary categorical variables are represented using a single encoded column, and previously unseen categories are ignored to allow the pipeline to handle new observations.

`SeniorCitizen` is already represented as a binary numerical variable and is therefore passed through without additional transformation.

In [36]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [37]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            drop="if_binary"
        ), categorical_features)
    ],
    remainder="passthrough"
)

In [38]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## 4. Preprocessing Validation

The preprocessing pipeline was fitted exclusively on the training data to prevent data leakage.

After transformation, both the training and test sets contain the same set of features. Numerical variables were standardized, categorical variables were one-hot encoded, and `SeniorCitizen` was retained as an already binary numerical feature.

The preprocessing pipeline is now ready to be integrated with the machine learning models in the modeling stage.

In [39]:
print("Original train shape:", X_train.shape)
print("Processed train shape:", X_train_processed.shape)

print("Original test shape:", X_test.shape)
print("Processed test shape:", X_test_processed.shape)

preprocessor.get_feature_names_out()

Original train shape: (5634, 19)
Processed train shape: (5634, 40)
Original test shape: (1409, 19)
Processed test shape: (1409, 40)


array(['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges',
       'cat__MultipleLines_No', 'cat__MultipleLines_No phone service',
       'cat__MultipleLines_Yes', 'cat__InternetService_DSL',
       'cat__InternetService_Fiber optic', 'cat__InternetService_No',
       'cat__OnlineSecurity_No',
       'cat__OnlineSecurity_No internet service',
       'cat__OnlineSecurity_Yes', 'cat__OnlineBackup_No',
       'cat__OnlineBackup_No internet service', 'cat__OnlineBackup_Yes',
       'cat__DeviceProtection_No',
       'cat__DeviceProtection_No internet service',
       'cat__DeviceProtection_Yes', 'cat__TechSupport_No',
       'cat__TechSupport_No internet service', 'cat__TechSupport_Yes',
       'cat__StreamingTV_No', 'cat__StreamingTV_No internet service',
       'cat__StreamingTV_Yes', 'cat__StreamingMovies_No',
       'cat__StreamingMovies_No internet service',
       'cat__StreamingMovies_Yes', 'cat__Contract_Month-to-month',
       'cat__Contract_One year', 'cat__Contract_Two yea